> **Versión corregida (v2.12, ejecución verificada)** — generada a partir de `notebooks/09_evaluacion_integral_equidad_v2.ipynb` sin sobrescribirlo.
>
> Cambios aplicados:
> - Celda de configuración del repositorio sustituida por el bootstrap portable (`PROJECT_ROOT`/`REPO_ROOT`).
> - Se eliminan `from google.colab import drive`, `drive.mount(...)` y las rutas fijas `/content/drive/MyDrive/PIDA-RL-Diabetes`; resultados en `RESULTS_V2_DIR` (`results/capacidad_v2`, o `results/capacidad_v2_prueba` en modo prueba).
> - Modo prueba: submuestra estratificada por `categoria_riesgo` (sin crear columnas) y capacidad escalada proporcionalmente para conservar la presión de recursos por persona.
> - Modo prueba: los escenarios de sensibilidad 50/100/150 se escalan con el mismo factor que la cohorte.
> - Las rutas de los tres modelos se toman de `RESULTS_V2_DIR`.
>
> Rutas: `PROJECT_ROOT` = datos/modelos/resultados (Drive en Colab, o `PIDA_PROJECT_ROOT`); `REPO_ROOT` = código. Modo prueba: `PIDA_MODO_PRUEBA=1` (cohorte pequeña y pocos episodios).


# 09 — Evaluación integral y equidad operativa V2

Compara las seis políticas con semillas comunes, analiza sensibilidad a capacidad y describe cobertura, intensidad, ajustes y eventos por subgrupo.

**Alcance:** simulación metodológica. Las brechas son diagnósticos operativos y no demuestran equidad ni efectividad clínica. Solo se conservan las categorías de riesgo **bajo, medio y alto**.


In [ ]:
# === Bootstrap portable: PROJECT_ROOT (corrección v2.12) ===
# PROJECT_ROOT = raíz persistente de datos, modelos y resultados.
# REPO_ROOT    = copia local del código (repositorio GitHub).
# Prioridad: variables de entorno PIDA_PROJECT_ROOT / PIDA_REPO_ROOT,
# después Google Colab + Drive y, por último, ejecución local.
from pathlib import Path
from datetime import datetime, timezone
import os
import sys
import json
import importlib
import importlib.util
import subprocess

PROJECT_NAME = 'pida-rl-datos-publicos'

REPO_URL = 'https://github.com/JssSalas/pida-rl-datos-publicos.git'
BRANCH = os.environ.get('PIDA_BRANCH', 'feature/entorno-capacidad-v2')
PROJECT_FOLDER_NAME = 'PIDA-RL-Diabetes'
IN_COLAB = 'google.colab' in sys.modules

DRIVE_ROOT = None
if os.environ.get('PIDA_PROJECT_ROOT'):
    PROJECT_ROOT = Path(os.environ['PIDA_PROJECT_ROOT']).expanduser().resolve()
elif IN_COLAB:
    DRIVE_ROOT = next(
        (p for p in [Path('/content/drive'), Path('/content/gdrive')] if (p / 'MyDrive').is_dir()),
        None,
    )
    if DRIVE_ROOT is None:
        from google.colab import drive
        mount_point = Path('/content/drive')
        if mount_point.exists() and any(mount_point.iterdir()):
            mount_point = Path('/content/gdrive')
        drive.mount(str(mount_point), force_remount=False)
        DRIVE_ROOT = mount_point
    PROJECT_ROOT = DRIVE_ROOT / 'MyDrive' / PROJECT_FOLDER_NAME
else:
    PROJECT_ROOT = (Path.cwd() / PROJECT_FOLDER_NAME).resolve()
PROJECT_ROOT.mkdir(parents=True, exist_ok=True)


def _buscar_repo(inicio):
    for candidato in [inicio, *inicio.parents]:
        if (candidato / 'src' / 'environment' / 'diabetes_followup_env.py').is_file():
            return candidato
    return None


if os.environ.get('PIDA_REPO_ROOT'):
    REPO_ROOT = Path(os.environ['PIDA_REPO_ROOT']).expanduser().resolve()
else:
    REPO_ROOT = _buscar_repo(Path.cwd().resolve()) or (
        Path('/content/pida-rl-datos-publicos') if IN_COLAB
        else Path.cwd().resolve() / 'pida-rl-datos-publicos'
    )
if not REPO_ROOT.exists():
    subprocess.check_call(['git', 'clone', '--branch', BRANCH, '--single-branch', REPO_URL, str(REPO_ROOT)])
elif not (REPO_ROOT / 'src').is_dir():
    raise RuntimeError(f'REPO_ROOT no contiene la carpeta src: {REPO_ROOT}')
elif IN_COLAB and (REPO_ROOT / '.git').is_dir():
    # En Colab se alinea la copia temporal con la rama de trabajo (sin push ni commit).
    for args in (['fetch', 'origin', BRANCH], ['checkout', BRANCH], ['pull', '--ff-only', 'origin', BRANCH]):
        subprocess.run(['git', '-C', str(REPO_ROOT), *args], check=False)

SRC_ROOT = REPO_ROOT / 'src'
for ruta in (str(REPO_ROOT), str(SRC_ROOT)):
    if ruta not in sys.path:
        sys.path.insert(0, ruta)
os.chdir(REPO_ROOT)

# Estructura persistente común (antes: rutas fijas /content/drive/MyDrive/PIDA-RL-Diabetes).
PERSISTENT_ROOT = PROJECT_ROOT  # alias de compatibilidad con los notebooks originales
MYDRIVE_ROOT = PROJECT_ROOT.parent
DATA_ROOT = PROJECT_ROOT / 'data'
RAW_DATA_DIR = DATA_ROOT / 'raw'
INTERIM_DATA_DIR = DATA_ROOT / 'interim'
PROCESSED_DATA_DIR = DATA_ROOT / 'processed'
METADATA_DIR = DATA_ROOT / 'metadata'
SPECIFICATIONS_DIR = DATA_ROOT / 'specifications'
MODELS_DIR = PROJECT_ROOT / 'models'
RESULTS_DIR = PROJECT_ROOT / 'results'
COHORT_PATH = PROCESSED_DATA_DIR / 'cohorte_diabetes.csv'

# Modo prueba: cohorte pequeña y pocos episodios (PIDA_MODO_PRUEBA=1).
MODO_PRUEBA = os.environ.get('PIDA_MODO_PRUEBA', '0') == '1'
N_COHORTE_PRUEBA = int(os.environ.get('PIDA_N_COHORTE', '120'))
SEMILLA_MUESTRA_PRUEBA = 2026
RESULTS_V2_DIR = RESULTS_DIR / ('capacidad_v2_prueba' if MODO_PRUEBA else 'capacidad_v2')


def muestra_cohorte_prueba(cohorte, n=None, semilla=SEMILLA_MUESTRA_PRUEBA):
    """Devuelve (cohorte, factor). En modo prueba toma una submuestra
    estratificada por categoria_riesgo (bajo/medio/alto); no crea columnas."""
    n = N_COHORTE_PRUEBA if n is None else int(n)
    if not MODO_PRUEBA or len(cohorte) <= n:
        return cohorte, 1.0
    fraccion = n / len(cohorte)
    muestra = (
        cohorte.groupby('categoria_riesgo', group_keys=False)
        .sample(frac=fraccion, random_state=semilla)
        .sort_values('FOLIO_INT')
        .reset_index(drop=True)
    )
    return muestra, len(muestra) / len(cohorte)


def escalar_capacidad(capacidad, factor):
    """Mantiene la presión de capacidad (recursos/persona) al reducir la cohorte."""
    return capacidad if factor >= 1.0 else max(1, int(round(capacidad * factor)))

print('Entorno:', 'Google Colab' if IN_COLAB else 'local')
print('PROJECT_ROOT (datos/resultados):', PROJECT_ROOT)
print('REPO_ROOT (código):', REPO_ROOT)
print('Modo prueba:', MODO_PRUEBA)

for _mod, _pip in [('gymnasium', 'gymnasium'), ('pandas', 'pandas')]:
    if importlib.util.find_spec(_mod) is None:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--quiet', _pip])
try:
    print('Commit evaluado:', subprocess.check_output(['git', '-C', str(REPO_ROOT), 'rev-parse', 'HEAD'], text=True).strip())
except Exception as exc:  # copia sin .git
    print('Commit evaluado: no disponible', exc)


## Configuración común

La evaluación principal y la sensibilidad reutilizan costos, horizonte y semillas. Las capacidades son escenarios operativos explícitos, no estimaciones institucionales.


In [ ]:
CAPACIDAD_MENSUAL = 100
CAPACIDADES_SENSIBILIDAD = (50, 100, 150)
COSTOS_ACCION = {0: 0, 1: 1, 2: 3, 3: 5}
HORIZONTE_MESES = 12
SEMILLAS_EVALUACION = (42, 123, 2026)


## Cohorte y modelos

Ejecute previamente los notebooks 06, 07 y 08. Este bloque no reentrena ni fabrica pacientes; carga modelos persistidos y exige que estén presentes.


In [ ]:
import pandas as pd
COHORT_PATH = PROCESSED_DATA_DIR / "cohorte_diabetes.csv"
OUTPUT_DIR = RESULTS_V2_DIR / "evaluacion_integral"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
cohorte = pd.read_csv(COHORT_PATH, dtype={"FOLIO_INT": "string"})

# Modo prueba: cohorte pequeña estratificada y capacidad proporcional.
N_COHORTE_COMPLETA = len(cohorte)
cohorte, FACTOR_COHORTE = muestra_cohorte_prueba(cohorte)
CAPACIDAD_MENSUAL = escalar_capacidad(CAPACIDAD_MENSUAL, FACTOR_COHORTE)
CAPACIDADES_SENSIBILIDAD = tuple(escalar_capacidad(c, FACTOR_COHORTE) for c in CAPACIDADES_SENSIBILIDAD)
assert set(cohorte['categoria_riesgo'].unique()) <= {'bajo', 'medio', 'alto'}
print(f'Cohorte usada: {len(cohorte)} de {N_COHORTE_COMPLETA} perfiles | capacidad mensual: {CAPACIDAD_MENSUAL}')
print(cohorte['categoria_riesgo'].value_counts().to_dict())
assert set(cohorte["categoria_riesgo"].unique()) <= {"bajo", "medio", "alto"}
MODELOS = {
    "Q-learning": RESULTS_V2_DIR / "q_learning" / "q_learning_v2.npz",
    "DQN": RESULTS_V2_DIR / "dqn" / "dqn_v2.npz",
    "PPO": RESULTS_V2_DIR / "ppo" / "ppo_v2.npz",
}
faltantes = [str(ruta) for ruta in MODELOS.values() if not ruta.exists()]
if faltantes:
    raise FileNotFoundError("Ejecute notebooks 06-08. Modelos faltantes: " + ", ".join(faltantes))


## Políticas comparables

Las seis políticas usan exactamente la misma cohorte, dinámica, costos, horizonte y semillas.


In [ ]:
from src.environment.diabetes_followup_env import DiabetesFollowUpEnv
from src.policies.baselines import POLITICAS_BASE
from src.policies.q_learning import QLearningAgent, crear_politica_q_learning
from src.policies.dqn import DQNAgent, crear_politica_dqn
from src.policies.ppo import PPOAgent, crear_politica_ppo
from src.evaluation.evaluate_policies import evaluar_politicas
from src.evaluation.integral import construir_reporte_integral, exportar_reporte_integral, sensibilidad_capacidad

def crear_entorno(capacidad=CAPACIDAD_MENSUAL):
    return DiabetesFollowUpEnv(cohorte, capacidad, COSTOS_ACCION, HORIZONTE_MESES)

politicas = dict(POLITICAS_BASE)
politicas["Q-learning"] = crear_politica_q_learning(QLearningAgent.cargar(MODELOS["Q-learning"], seed=20_000))
politicas["DQN"] = crear_politica_dqn(DQNAgent.cargar(MODELOS["DQN"], seed=20_000))
politicas["PPO"] = crear_politica_ppo(PPOAgent.cargar(MODELOS["PPO"], seed=20_000))
assert len(politicas) == 6


## Evaluación principal

Se conservan acciones propuestas y ejecutadas. El análisis de subgrupos se realiza después de la decisión para no cambiar la información disponible para las políticas.


In [ ]:
resultados = evaluar_politicas(lambda: crear_entorno(), politicas, SEMILLAS_EVALUACION)
sensibilidad = sensibilidad_capacidad(crear_entorno, politicas, CAPACIDADES_SENSIBILIDAD, SEMILLAS_EVALUACION)
reporte = construir_reporte_integral(resultados, cohorte, sensibilidad)
detalle = reporte["detalle"]
assert set(detalle["categoria_riesgo"]) <= {"bajo", "medio", "alto"}
assert (detalle["recursos_despues"] >= 0).all()
assert (detalle["recursos_usados_mes"] <= detalle["capacidad_mensual"]).all()
rutas = exportar_reporte_integral(reporte, OUTPUT_DIR)
display(reporte["resumen_algoritmos"].round(4))
display(reporte["estabilidad_semillas"].round(4))


## Equidad operativa

Las tablas muestran resultados por sexo codificado en la fuente, grupos de edad y nivel de riesgo. No se interpreta una diferencia como discriminación ni como efecto causal; primero debe revisarse tamaño de grupo, incertidumbre, codificación y mecanismo del simulador.


In [ ]:
display(reporte["metricas_subgrupos"].round(4))
display(reporte["brechas_equidad"].round(4))
display(reporte["sensibilidad_resumen"].round(4))
print("Archivos exportados:", len(rutas), OUTPUT_DIR)


## Criterios de lectura responsable

- No seleccione una política únicamente por recompensa.
- Revise cobertura de riesgo alto, eventos, recursos, ajustes, estabilidad y brechas conjuntamente.
- Compare acción propuesta contra ejecutada para identificar demanda no atendida.
- Si cambian cohorte, recompensa, costos o capacidad, reentrene Q-learning, DQN y PPO.
- La validación con simulación no sustituye validación clínica, ética ni institucional.
